# Pertemuan 4 — Bagian 10: Diagnosis kualitas data proyek tim

**Dataset:** COVID-19 Indonesia (Hendratno, Kaggle), berkas `covid_19_indonesia_time_series_all.csv`

> **Aturan hari ini: jangan memperbaiki apa pun.** Notebook ini hanya mendiagnosis. `df_tim` tidak pernah diubah.
> Kalau sebuah pemeriksaan butuh kolom tanggal yang sudah dikonversi, konversinya dilakukan pada salinan terpisah.

Hasil bagian ini menjadi bagian pertama M2 (Data Quality Report).

**Pertanyaan analisis tim** (dari Pertemuan 3), yang dipakai untuk menentukan tingkat keparahan: kolom
`Date`, `Location`, `Location Level`, `New Cases`, `New Deaths`, `Population`, `Population Density`, dan `Island`.
Masalah pada kolom-kolom ini lebih parah daripada masalah pada kolom yang tidak dipakai.

## 10.1 — Muat dataset proyek tim

In [1]:
import os, shutil
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)

NAMA_FILE = 'covid_19_indonesia_time_series_all.csv'
RAW = f'data/raw/{NAMA_FILE}'
KOLOM_WAJIB = ['Date', 'Location', 'Location Level', 'New Cases', 'New Deaths',
               'Population', 'Population Density', 'Island']

os.makedirs('data/raw', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# Unggah langsung dari laptop (dilewati kalau berkas sudah ada di data/raw/)
if not os.path.exists(RAW):
    from google.colab import files
    print("Klik 'Choose Files' lalu pilih", NAMA_FILE, 'dari laptop Anda:')
    unggahan = files.upload()
    assert unggahan, 'Tidak ada berkas yang dipilih. Jalankan sel ini lagi.'
    shutil.move(next(iter(unggahan)), RAW)

df_tim = pd.read_csv(RAW)

# Validasi: pastikan ini berkas lengkap, bukan versi yang sudah dipotong
hilang = [c for c in KOLOM_WAJIB if c not in df_tim.columns]
if hilang or len(df_tim) < 20000:
    os.remove(RAW)
    raise ValueError(f'Berkas salah: {len(df_tim)} baris, {df_tim.shape[1]} kolom, kolom wajib hilang: {hilang}. '
                     'Berkas dihapus; jalankan sel ini lagi dan pilih berkas asli dari Kaggle.')

df_tim.info()

<class 'pandas.DataFrame'>
RangeIndex: 31822 entries, 0 to 31821
Data columns (total 38 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Date                         31822 non-null  str    
 1   Location ISO Code            31822 non-null  str    
 2   Location                     31822 non-null  str    
 3   New Cases                    31822 non-null  int64  
 4   New Deaths                   31822 non-null  int64  
 5   New Recovered                31822 non-null  int64  
 6   New Active Cases             31822 non-null  int64  
 7   Total Cases                  31822 non-null  int64  
 8   Total Deaths                 31822 non-null  int64  
 9   Total Recovered              31822 non-null  int64  
 10  Total Active Cases           31822 non-null  int64  
 11  Location Level               31822 non-null  str    
 12  City or Regency              0 non-null      float64
 13  Province                   

In [2]:
# Alat bantu dari workbook (Bagian 4 dan Bagian 9), disesuaikan nama datanya
def pola_kosong(data, kolom_kosong, kolom_pembanding):
    """Persentase kekosongan kolom_kosong untuk tiap nilai kolom_pembanding."""
    return (data.groupby(kolom_pembanding, observed=True)[kolom_kosong]
                .apply(lambda s: round(s.isna().mean() * 100, 1))
                .sort_values(ascending=False)
                .to_frame('persen_kosong'))

temuan = []

def catat(dimensi, kolom, masalah, jumlah, keparahan, rencana):
    temuan.append({
        'dimensi'   : dimensi,
        'kolom'     : kolom,
        'masalah'   : masalah,
        'jumlah'    : int(jumlah),
        'persen'    : round(jumlah / len(df_tim) * 100, 2),
        'keparahan' : keparahan,          # Tinggi / Sedang / Rendah
        'rencana'   : rencana,            # diisi lengkap pada pertemuan 5
    })

# salinan bantu (df_tim sendiri tidak diubah)
tgl = pd.to_datetime(df_tim['Date'], format='%m/%d/%Y')
nasional = df_tim['Location Level'].eq('Country')
provinsi = df_tim['Location Level'].eq('Province')
kolom_teks = [c for c in df_tim.columns if not pd.api.types.is_numeric_dtype(df_tim[c])]
kolom_angka = [c for c in df_tim.columns if pd.api.types.is_numeric_dtype(df_tim[c])]
print('Alat bantu siap. Kolom teks:', len(kolom_teks), '| kolom angka:', len(kolom_angka))

Alat bantu siap. Kolom teks: 12 | kolom angka: 26


## 10.2 — Sembilan pemeriksaan

Urutannya mengikuti lima dimensi: kelengkapan, keunikan, konsistensi, validitas, akurasi.

### Pemeriksaan 1 — Potret awal

In [3]:
print('Ukuran:', df_tim.shape[0], 'baris,', df_tim.shape[1], 'kolom')
print('Satuan amatan: satu baris = satu wilayah (provinsi atau nasional) pada satu tanggal')
print('Baris provinsi:', int(provinsi.sum()), '| baris nasional:', int(nasional.sum()))
print()
print(df_tim[kolom_angka].describe().T.round(1).to_string())
print()
print(df_tim[kolom_teks].describe().T.to_string())

Ukuran: 31822 baris, 38 kolom
Satuan amatan: satu baris = satu wilayah (provinsi atau nasional) pada satu tanggal
Baris provinsi: 30893 | baris nasional: 929

                               count        mean         std       min        25%        50%        75%          max
New Cases                    31822.0       402.3      2320.6       0.0        3.0       27.0      130.0      64718.0
New Deaths                   31822.0         9.9        64.1       0.0        0.0        0.0        3.0       2069.0
New Recovered                31822.0       390.4      2199.9       0.0        2.0       20.0      123.0      61361.0
New Active Cases             31822.0         2.0      1219.5  -29938.0      -12.0        0.0       19.0      39165.0
Total Cases                  31822.0    159450.0    626443.5       1.0     5223.2    23596.5    69927.8    6405044.0
Total Deaths                 31822.0      4564.8     17693.7       0.0      128.0      565.5     2189.0     157876.0
Total Recovered       

**Temuan:** 31.822 baris dan 38 kolom. Tiga kolom teks seharusnya bukan teks: `Date`, `Case Fatality Rate`,
dan `Case Recovered Rate`. Baris provinsi (30.893) bercampur dengan baris nasional (929).

### Pemeriksaan 2 — Nilai kosong sejati

In [4]:
kosong = df_tim.isna().sum()
tabel_kosong = (pd.DataFrame({'jumlah_kosong': kosong, 'persen': (kosong / len(df_tim) * 100).round(1)})
                .query('jumlah_kosong > 0').sort_values('persen', ascending=False))
print(tabel_kosong.to_string())
print()
print('Kolom dengan kosong di atas 5%:', tabel_kosong.query('persen > 5').index.tolist())

                             jumlah_kosong  persen
City or Regency                      31822   100.0
Special Status                       27264    85.7
Growth Factor of New Deaths           3447    10.8
Growth Factor of New Cases            1939     6.1
Province                               929     2.9
Island                                 929     2.9
Total Rural Villages                   929     2.9
Time Zone                              929     2.9
Total Cities                           901     2.8
Total Urban Villages                   904     2.8

Kolom dengan kosong di atas 5%: ['City or Regency', 'Special Status', 'Growth Factor of New Deaths', 'Growth Factor of New Cases']


**Temuan:** sembilan kolom punya nilai kosong. Empat di antaranya di atas 5% dan akan dinilai pola kosongnya
di Pemeriksaan 4: `City or Regency`, `Special Status`, dan dua kolom `Growth Factor`.

### Pemeriksaan 3 — Placeholder teks dan angka

In [5]:
# 3a. placeholder teks
PENANDA_TEKS = {'', '-', '--', 'n/a', 'na', 'null', 'none', 'nan', 'unknown', 'tidak diketahui', 'tidak ada', '?'}
ada = False
for kol in kolom_teks:
    s = df_tim[kol].dropna().astype(str).str.strip().str.lower()
    n = int(s.isin(PENANDA_TEKS).sum())
    if n:
        ada = True
        print(f'{kol}: {n} baris berisi penanda kosong')
print('Placeholder teks:', 'ditemukan' if ada else 'tidak ditemukan pada semua kolom teks')

# 3b. placeholder angka: lihat nilai ekstrem tiap kolom angka
print()
ekstrem = pd.DataFrame({'min': df_tim[kolom_angka].min(), 'maks': df_tim[kolom_angka].max()})
print(ekstrem.to_string())

Placeholder teks: tidak ditemukan pada semua kolom teks

                                       min          maks
New Cases                         0.000000  6.471800e+04
New Deaths                        0.000000  2.069000e+03
New Recovered                     0.000000  6.136100e+04
New Active Cases             -29938.000000  3.916500e+04
Total Cases                       1.000000  6.405044e+06
Total Deaths                      0.000000  1.578760e+05
Total Recovered                   0.000000  6.218708e+06
Total Active Cases            -2343.000000  5.861130e+05
City or Regency                        NaN           NaN
Total Regencies                   1.000000  4.160000e+02
Total Cities                      1.000000  9.800000e+01
Total Districts                  44.000000  7.230000e+03
Total Urban Villages             35.000000  8.488000e+03
Total Rural Villages            275.000000  7.495300e+04
Area (km2)                      664.000000  1.916907e+06
Population                   64

In [6]:
# 3c. nilai yang biasa dipakai sebagai placeholder: berapa kali muncul di tiap kolom angka?
KANDIDAT = [-999, -1, 9999, 999]
baris = []
for kol in kolom_angka:
    for v in KANDIDAT:
        n = int((df_tim[kol] == v).sum())
        if n:
            baris.append({'kolom': kol, 'nilai': v, 'jumlah': n})
print(pd.DataFrame(baris).to_string(index=False))

             kolom  nilai  jumlah
         New Cases    999       1
     New Recovered    999       4
  New Active Cases     -1    1149
  New Active Cases    999       1
       Total Cases   9999       2
      Total Deaths    999       3
   Total Recovered    999       2
Total Active Cases     -1      63
Total Active Cases   9999       1
Total Active Cases    999       2


In [7]:
# 3d. apakah nilai-nilai itu nyata atau placeholder? Uji dengan hubungan antarkolom.
d = df_tim.assign(tgl=tgl).sort_values(['Location', 'tgl'])

# (i) -1 pada New Active Cases: kalau nyata, harus sama dengan New Cases - New Deaths - New Recovered
selisih_aktif = (d['New Active Cases'] != d['New Cases'] - d['New Deaths'] - d['New Recovered']).sum()
print('New Active Cases tidak sama dengan New Cases - New Deaths - New Recovered:', int(selisih_aktif), 'baris')

# (ii) 999 dan 9999 pada kolom Total: kalau nyata, harus berada di antara hari sebelum dan sesudahnya
for kol in ['Total Cases', 'Total Deaths', 'Total Recovered']:
    sebelum = d.groupby('Location')[kol].shift()
    sesudah = d.groupby('Location')[kol].shift(-1)
    m = d[kol].isin([999, 9999])
    wajar = ((sebelum <= d[kol]) & (d[kol] <= sesudah))[m]
    print(f'{kol}: {int(m.sum())} baris bernilai 999/9999; yang wajar terhadap tetangganya: {int(wajar.sum())}')

New Active Cases tidak sama dengan New Cases - New Deaths - New Recovered: 0 baris
Total Cases: 2 baris bernilai 999/9999; yang wajar terhadap tetangganya: 2


Total Deaths: 3 baris bernilai 999/9999; yang wajar terhadap tetangganya: 3
Total Recovered: 2 baris bernilai 999/9999; yang wajar terhadap tetangganya: 2


**Temuan:** tidak ada placeholder. Angka `-1`, `999`, dan `9999` yang tampak mencurigakan ternyata nilai sah:
`-1` pada `New Active Cases` cocok dengan hitungan (kasus aktif turun 1), dan `999`/`9999` pada kolom `Total`
berada tepat di antara hari sebelum dan sesudahnya (total kumulatif yang kebetulan melewati angka itu). Angka harian seperti 999 pada `New Cases` juga didukung Pemeriksaan 9: selisih total antarhari selalu sama dengan `New Cases`, jadi placeholder tidak mungkin bersembunyi di sana.

In [8]:
# 3e. angka 0 pada New Cases: nol sejati atau hari tanpa pelaporan?
p = df_tim[provinsi].assign(tgl=tgl[provinsi])
p['nol'] = p['New Cases'].eq(0)
print('Baris provinsi dengan New Cases = 0:', int(p['nol'].sum()), f"({p['nol'].mean()*100:.1f}%)")
p22 = p[p['tgl'].dt.year == 2022]
print('Khusus 2022:', int(p22['nol'].sum()), f"({p22['nol'].mean()*100:.1f}%)")
print()
print('Persen hari nol menurut hari dalam pekan (2022):')
print((p22.groupby(p22['tgl'].dt.day_name())['nol'].mean() * 100).round(1)
      .reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']).to_string())
per_prov = (p22.groupby('Location')['nol'].mean() * 100).round(1).sort_values(ascending=False)
print()
print('Persen hari nol menurut provinsi (2022), tertinggi:', per_prov.head(4).to_dict())
print('Persen hari nol menurut provinsi (2022), terendah:', per_prov.tail(3).to_dict())

Baris provinsi dengan New Cases = 0: 4098 (13.3%)
Khusus 2022: 1565 (17.8%)

Persen hari nol menurut hari dalam pekan (2022):
tgl
Monday       22.8
Tuesday      16.1
Wednesday    15.4
Thursday     14.3
Friday       17.2
Saturday     16.6
Sunday       22.4

Persen hari nol menurut provinsi (2022), tertinggi: {'Bengkulu': 55.4, 'Gorontalo': 54.5, 'Sulawesi Barat': 52.1, 'Maluku Utara': 48.4}
Persen hari nol menurut provinsi (2022), terendah: {'Banten': 0.0, 'DKI Jakarta': 0.0, 'Jawa Barat': 0.0}


**Temuan:** hari bernilai 0 terkonsentrasi pada provinsi kecil (Bengkulu, Gorontalo, Sulawesi Barat lebih dari separuh harinya)
dan hampir tidak ada di Jakarta, Jawa Barat, dan Banten. Persentasenya juga sedikit lebih tinggi pada Senin dan Minggu.
Dari data saja **tidak bisa dibedakan** antara nol sejati (memang tidak ada kasus baru) dan hari tanpa pelaporan.
Total kumulatif tetap konsisten dengan `New Cases` (Pemeriksaan 9), jadi jumlah selama periode panjang tidak bias,
tetapi analisis harian pada provinsi kecil perlu hati-hati.

### Pemeriksaan 4 — Pola kekosongan: MCAR, MAR, atau MNAR

Ingat: **MAR bisa dibuktikan dari data** (kosongnya mengikuti kolom lain). MCAR dan MNAR tidak bisa dibedakan dari data saja.

In [9]:
df_bantu = df_tim.assign(tgl=tgl).sort_values(['Location', 'tgl']).copy()
g = df_bantu.groupby('Location')
pertama = g.cumcount().eq(0)
bagi_nol_kasus = g['New Cases'].shift().eq(0) & df_bantu['New Cases'].gt(0)       # x dibagi 0
bagi_nol_mati = g['New Deaths'].shift().eq(0) & df_bantu['New Deaths'].gt(0)
df_bantu['sebab_kasus'] = np.where(pertama, 'hari pertama lokasi',
                          np.where(bagi_nol_kasus, 'kemarin 0 dan hari ini lebih dari 0', 'keadaan lain'))
df_bantu['sebab_kematian'] = np.where(pertama, 'hari pertama lokasi',
                             np.where(bagi_nol_mati, 'kemarin 0 dan hari ini lebih dari 0', 'keadaan lain'))

print('--- Special Status kosong menurut provinsi (hanya yang TIDAK 100% kosong) ---')
sp = pola_kosong(df_bantu, 'Special Status', 'Location')
print(sp[sp['persen_kosong'] < 100].to_string())
print('Jumlah lokasi yang 100% kosong:', int((sp['persen_kosong'] == 100).sum()), 'dari', len(sp))

print()
print('--- Growth Factor of New Cases kosong menurut keadaan kasus baru ---')
print(pola_kosong(df_bantu, 'Growth Factor of New Cases', 'sebab_kasus').to_string())
print()
print('--- Growth Factor of New Deaths kosong menurut keadaan kematian baru ---')
print(pola_kosong(df_bantu, 'Growth Factor of New Deaths', 'sebab_kematian').to_string())

print()
print('--- Province, Island, Time Zone kosong menurut Location Level ---')
for kol in ['Province', 'Island', 'Time Zone']:
    print(kol, pola_kosong(df_bantu, kol, 'Location Level')['persen_kosong'].to_dict())

print()
print('--- kolom jumlah wilayah: lokasi yang kosong ---')
for kol in ['Total Rural Villages', 'Total Cities', 'Total Urban Villages']:
    x = pola_kosong(df_bantu[df_bantu['Location Level'] == 'Province'], kol, 'Location')
    print(kol, x[x['persen_kosong'] > 0]['persen_kosong'].to_dict())

print()
print('City or Regency kosong:', round(df_tim['City or Regency'].isna().mean() * 100, 1), '%')

--- Special Status kosong menurut provinsi (hanya yang TIDAK 100% kosong) ---
                            persen_kosong
Location                                 
Aceh                                  0.0
Daerah Istimewa Yogyakarta            0.0
DKI Jakarta                           0.0
Papua Barat                           0.0
Papua                                 0.0
Jumlah lokasi yang 100% kosong: 30 dari 35

--- Growth Factor of New Cases kosong menurut keadaan kasus baru ---
                                     persen_kosong
sebab_kasus                                       
hari pertama lokasi                          100.0
kemarin 0 dan hari ini lebih dari 0          100.0
keadaan lain                                   0.0

--- Growth Factor of New Deaths kosong menurut keadaan kematian baru ---
                                     persen_kosong
sebab_kematian                                    
hari pertama lokasi                          100.0
kemarin 0 dan hari ini lebih dari

Province {'Country': 100.0, 'Province': 0.0}
Island {'Country': 100.0, 'Province': 0.0}
Time Zone {'Country': 100.0, 'Province': 0.0}

--- kolom jumlah wilayah: lokasi yang kosong ---
Total Rural Villages {'DKI Jakarta': 100.0}


Total Cities {'Sulawesi Barat': 100.0}
Total Urban Villages {'Aceh': 100.0}

City or Regency kosong: 100.0 %


In [10]:
def pct(kol):
    return round(df_tim[kol].isna().mean() * 100, 1)

pola_df = pd.DataFrame([
    ('City or Regency', pct('City or Regency'), 'Tidak dapat dinilai (kosong struktural)',
     'Kosong 100%, jadi tidak ada baris terisi untuk dibandingkan. Deskripsi dataset di Kaggle menyebut data tingkat kota hanya disediakan bila diperlukan pada provinsi tertentu, sehingga kolom ini tampaknya memang tidak pernah diisi.'),
    ('Special Status', pct('Special Status'), 'MAR',
     'Terbukti dari data: hanya 5 provinsi yang terisi (Aceh, DKI Jakarta, DI Yogyakarta, Papua, Papua Barat), dan di provinsi lain 100% kosong. Kosongnya mengikuti kolom Location, yaitu provinsi tanpa status khusus.'),
    ('Growth Factor of New Deaths', pct('Growth Factor of New Deaths'), 'MAR',
     'Terbukti dari data: kosong 100% pada hari pertama lokasi dan bila kematian baru kemarin 0 sedangkan hari ini lebih dari 0 (pembagian dengan nol), dan 0% pada keadaan lain. Nilainya tidak terdefinisi, bukan hilang.'),
    ('Growth Factor of New Cases', pct('Growth Factor of New Cases'), 'MAR',
     'Sama seperti kolom kematian: kosong 100% pada hari pertama lokasi dan bila kasus baru kemarin 0 sedangkan hari ini lebih dari 0 (pembagian dengan nol), dan 0% pada keadaan lain.'),
    ('Province, Island, Time Zone', pct('Province'), 'MAR (di bawah 5%, sebagai informasi)',
     'Kosong 100% pada baris nasional (Location Level = Country) dan 0% pada baris provinsi.'),
    ('Total Rural Villages', pct('Total Rural Villages'), 'MAR (di bawah 5%, sebagai informasi)',
     'Kosong hanya untuk DKI Jakarta. Jakarta hanya memiliki kelurahan, tidak memiliki desa.'),
    ('Total Cities', pct('Total Cities'), 'MAR (di bawah 5%, sebagai informasi)',
     'Kosong hanya untuk Sulawesi Barat, provinsi yang tidak memiliki kota.'),
    ('Total Urban Villages', pct('Total Urban Villages'), 'MAR (di bawah 5%, sebagai informasi)',
     'Kosong hanya untuk Aceh. Alasannya belum dikonfirmasi ke sumber data.'),
], columns=['kolom', 'persen_kosong', 'dugaan', 'dasar_dugaan'])
print(pola_df[['kolom', 'persen_kosong', 'dugaan']].to_string(index=False))

                      kolom  persen_kosong                                  dugaan
            City or Regency          100.0 Tidak dapat dinilai (kosong struktural)
             Special Status           85.7                                     MAR
Growth Factor of New Deaths           10.8                                     MAR
 Growth Factor of New Cases            6.1                                     MAR
Province, Island, Time Zone            2.9    MAR (di bawah 5%, sebagai informasi)
       Total Rural Villages            2.9    MAR (di bawah 5%, sebagai informasi)
               Total Cities            2.8    MAR (di bawah 5%, sebagai informasi)
       Total Urban Villages            2.8    MAR (di bawah 5%, sebagai informasi)

**Temuan:** tidak ada kolom yang dugaannya MNAR. Tiga kolom di atas 5% terbukti MAR dari data. Kolom `City or Regency`
kosong seluruhnya sehingga tidak bisa dinilai. Dugaan MCAR murni tidak muncul, karena setiap kekosongan punya penjelasan
dari kolom lain.

### Pemeriksaan 5 — Keunikan: duplikat persis dan duplikat nyaris

In [11]:
dup_persis = int(df_tim.duplicated().sum())
dup_kunci = int(df_tim.duplicated(subset=['Location', 'Date'], keep=False).sum())
# duplikat nyaris: kunci sama setelah spasi dan huruf dirapikan
kunci_rapi = df_tim['Location'].astype(str).str.strip().str.title() + '|' + df_tim['Date'].astype(str).str.strip()
dup_nyaris = int(kunci_rapi.duplicated(keep=False).sum()) - dup_kunci
print('Duplikat persis (seluruh kolom sama):', dup_persis)
print('Duplikat pada kunci (Location, Date):', dup_kunci)
print('Duplikat nyaris (kunci sama setelah dirapikan):', dup_nyaris)
print('Baris per lokasi, min dan maks:', int(df_tim.groupby('Location').size().min()), int(df_tim.groupby('Location').size().max()))

Duplikat persis (seluruh kolom sama): 0
Duplikat pada kunci (Location, Date): 0
Duplikat nyaris (kunci sama setelah dirapikan): 0


Baris per lokasi, min dan maks: 888 929


**Temuan:** tidak ada duplikat, baik persis maupun nyaris. Satu lokasi hanya punya satu baris pada tiap tanggal.

### Pemeriksaan 6 — Konsistensi teks dan atribut

In [12]:
baris = []
for kol in kolom_teks:
    s = df_tim[kol].dropna().astype(str)
    baris.append({'kolom': kol, 'unik_asli': s.nunique(), 'setelah_strip': s.str.strip().nunique(),
                  'setelah_strip_dan_title': s.str.strip().str.title().nunique()})
print(pd.DataFrame(baris).to_string(index=False))

print()
print('Atribut yang seharusnya tetap untuk satu provinsi: berapa provinsi punya lebih dari satu nilai?')
prov_df = df_tim[provinsi]
for kol in ['Population', 'Population Density', 'Area (km2)', 'Island', 'Time Zone']:
    print(f'  {kol}:', int((prov_df.groupby('Location')[kol].nunique() > 1).sum()))
print('Province sama dengan Location pada baris provinsi:', int((prov_df['Province'] == prov_df['Location']).sum()), 'dari', len(prov_df))
print('Jumlah nilai Island:', df_tim['Island'].nunique(), '| jumlah provinsi:', prov_df['Location'].nunique())

              kolom  unik_asli  setelah_strip  setelah_strip_dan_title
               Date        930            930                      930
  Location ISO Code         35             35                       35
           Location         35             35                       35
     Location Level          2              2                        2
           Province         34             34                       34
            Country          1              1                        1
          Continent          1              1                        1
             Island          7              7                        7
          Time Zone          3              3                        3
     Special Status          3              3                        3
 Case Fatality Rate       1328           1328                     1328
Case Recovered Rate       6170           6170                     6170

Atribut yang seharusnya tetap untuk satu provinsi: berapa provinsi punya leb

**Temuan:** tidak ada masalah penulisan (spasi, huruf besar-kecil, atau ejaan). Atribut tetap seperti penduduk, kepadatan,
dan pulau konsisten untuk setiap provinsi, jadi memakai `first` saat menggabungkan data per provinsi aman.
Masalah konsistensi yang ada bersifat struktur: baris nasional ("Indonesia") berada di kolom `Location` yang sama
dengan provinsi, sehingga menjumlahkan semua baris akan menghitung ganda.

### Pemeriksaan 7 — Tipe data yang salah

In [13]:
print('Kolom bukan angka:')
print(df_tim[kolom_teks].dtypes.astype(str).to_string())

print()
print('--- Date: apakah bulan/hari/tahun atau hari/bulan/tahun? ---')
bag = df_tim['Date'].str.split('/', expand=True).astype(int)
print('Bagian pertama, maks:', int(bag[0].max()), '| bagian kedua, maks:', int(bag[1].max()))
benar = pd.to_datetime(df_tim['Date'], format='%m/%d/%Y')
salah = pd.to_datetime(df_tim['Date'], format='%d/%m/%Y', errors='coerce')
print('Dibaca bulan/hari/tahun: rentang', benar.min().date(), 'sampai', benar.max().date(), '| bulan unik:', [int(x) for x in sorted(benar.dt.month.unique())])
print('Dibaca hari/bulan/tahun: gagal terbaca pada', int(salah.isna().sum()), 'baris')

print()
for kol in ['Case Fatality Rate', 'Case Recovered Rate']:
    print(kol, '-> semua berakhiran %:', bool(df_tim[kol].astype(str).str.endswith('%').all()), '| contoh:', df_tim[kol].iloc[500])

Kolom bukan angka:
Date                   str
Location ISO Code      str
Location               str
Location Level         str
Province               str
Country                str
Continent              str
Island                 str
Time Zone              str
Special Status         str
Case Fatality Rate     str
Case Recovered Rate    str

--- Date: apakah bulan/hari/tahun atau hari/bulan/tahun? ---
Bagian pertama, maks: 12 | bagian kedua, maks: 31
Dibaca bulan/hari/tahun: rentang 2020-03-01 sampai 2022-09-16 | bulan unik: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Dibaca hari/bulan/tahun: gagal terbaca pada 19178 baris

Case Fatality Rate -> semua berakhiran %: True | contoh: 10.00%
Case Recovered Rate -> semua berakhiran %: True | contoh: 10.00%


**Temuan:** tiga kolom bertipe teks. Format `Date` sudah diverifikasi: bagian pertama maksimal 12 dan bagian kedua maksimal 31,
jadi format yang benar adalah bulan/hari/tahun. Kalau dibaca hari/bulan/tahun, belasan ribu baris gagal terbaca,
sehingga tidak ada risiko tanggal tertukar diam-diam. Rentang hasil bacaan yang benar: 1 Maret 2020 sampai 16 September 2022.

### Pemeriksaan 8 — Aturan rentang (ditulis dari pengetahuan tentang data)

In [14]:
cfr = df_tim['Case Fatality Rate'].str.rstrip('%').astype(float)
crr = df_tim['Case Recovered Rate'].str.rstrip('%').astype(float)

hasil_aturan = []
def aturan(jenis, nama, dilanggar):
    hasil_aturan.append({'jenis': jenis, 'aturan': nama, 'baris_dilanggar': int(dilanggar.sum())})

aturan('Rentang', 'New Cases tidak boleh negatif (kasus baru tidak bisa kurang dari 0)', df_tim['New Cases'] < 0)
aturan('Rentang', 'New Deaths tidak boleh negatif', df_tim['New Deaths'] < 0)
aturan('Rentang', 'New Recovered tidak boleh negatif', df_tim['New Recovered'] < 0)
aturan('Rentang', 'Population harus lebih dari 0', df_tim['Population'] <= 0)
aturan('Rentang', 'Population Density harus lebih dari 0', df_tim['Population Density'] <= 0)
aturan('Rentang', 'Total Active Cases tidak boleh negatif (kasus aktif tidak bisa kurang dari 0)', df_tim['Total Active Cases'] < 0)
aturan('Rentang', 'Case Fatality Rate harus antara 0 dan 100 persen', (cfr < 0) | (cfr > 100))
aturan('Rentang', 'Case Recovered Rate harus antara 0 dan 100 persen', (crr < 0) | (crr > 100))
aturan('Rentang', 'Tanggal tidak boleh sebelum 1 Maret 2020 atau setelah tanggal pengambilan data (28 September 2026)',
       (tgl < '2020-03-01') | (tgl > '2026-09-28'))
print(pd.DataFrame(hasil_aturan).to_string(index=False))

  jenis                                                                                             aturan  baris_dilanggar
Rentang                                New Cases tidak boleh negatif (kasus baru tidak bisa kurang dari 0)                0
Rentang                                                                     New Deaths tidak boleh negatif                0
Rentang                                                                  New Recovered tidak boleh negatif                0
Rentang                                                                      Population harus lebih dari 0                0
Rentang                                                              Population Density harus lebih dari 0                0
Rentang                      Total Active Cases tidak boleh negatif (kasus aktif tidak bisa kurang dari 0)              479
Rentang                                                   Case Fatality Rate harus antara 0 dan 100 persen               43
Rentang 

**Temuan:** kolom yang dipakai pertanyaan analisis (`New Cases`, `New Deaths`, `Population`, `Population Density`, `Date`)
lolos semua aturan. Pelanggaran hanya ada pada kolom yang tidak dipakai: kasus aktif negatif (479 baris), tingkat kematian di atas 100%
(43 baris), dan tingkat kesembuhan di atas 100% (288 baris).

### Pemeriksaan 9 — Aturan hubungan antarkolom dan antarbaris

In [15]:
d = df_tim.assign(tgl=tgl).sort_values(['Location', 'tgl']).reset_index(drop=True)
kemarin = d.groupby('Location')['Total Cases'].shift()
ada_kemarin = kemarin.notna()

aturan('Hubungan', 'Total Deaths + Total Recovered tidak boleh melebihi Total Cases',
       df_tim['Total Deaths'] + df_tim['Total Recovered'] > df_tim['Total Cases'])
aturan('Hubungan', 'Total Deaths tidak boleh melebihi Total Cases', df_tim['Total Deaths'] > df_tim['Total Cases'])
aturan('Hubungan', 'Total Active Cases = Total Cases - Total Deaths - Total Recovered',
       df_tim['Total Active Cases'] != df_tim['Total Cases'] - df_tim['Total Deaths'] - df_tim['Total Recovered'])
aturan('Hubungan', 'Total Cases hari ini - kemarin = New Cases (pada lokasi yang sama)',
       ada_kemarin & ((d['Total Cases'] - kemarin) != d['New Cases']))
aturan('Hubungan', 'Total Cases tidak boleh turun dari hari sebelumnya', ada_kemarin & (d['Total Cases'] < kemarin))
aturan('Hubungan', 'Population Density = Population / Area (selisih paling banyak 1 persen)',
       (df_tim['Population'] / df_tim['Area (km2)'] - df_tim['Population Density']).abs() / df_tim['Population Density'] > 0.01)
aturan('Waktu', 'Tanggal pada satu lokasi harus berurutan tanpa lompatan',
       d.groupby('Location')['tgl'].diff().dt.days.gt(1))

# antarbaris: jumlah 34 provinsi harus sama dengan baris nasional pada tanggal yang sama
jml_prov = d[d['Location Level'] == 'Province'].groupby('tgl')['New Cases'].sum()
nas = d[d['Location Level'] == 'Country'].set_index('tgl')['New Cases']
banding = pd.concat([jml_prov.rename('jumlah_provinsi'), nas.rename('nasional')], axis=1).dropna()
tgl_beda = int((banding['jumlah_provinsi'] != banding['nasional']).sum())
hasil_aturan.append({'jenis': 'Hubungan', 'aturan': 'Jumlah New Cases 34 provinsi = New Cases baris nasional (tanggal yang sama)',
                     'baris_dilanggar': tgl_beda})

tabel_aturan = pd.DataFrame(hasil_aturan)
print(tabel_aturan.to_string(index=False))

   jenis                                                                                             aturan  baris_dilanggar
 Rentang                                New Cases tidak boleh negatif (kasus baru tidak bisa kurang dari 0)                0
 Rentang                                                                     New Deaths tidak boleh negatif                0
 Rentang                                                                  New Recovered tidak boleh negatif                0
 Rentang                                                                      Population harus lebih dari 0                0
 Rentang                                                              Population Density harus lebih dari 0                0
 Rentang                      Total Active Cases tidak boleh negatif (kasus aktif tidak bisa kurang dari 0)              479
 Rentang                                                   Case Fatality Rate harus antara 0 dan 100 persen               43


In [16]:
# seberapa besar bedanya? bandingkan total akhir
akhir = tgl.max() - pd.Timedelta(days=1)    # 15 September 2022, hari terakhir semua provinsi
tp = int(d[(d['Location Level'] == 'Province') & (d['tgl'] == akhir)]['Total Cases'].sum())
tn = int(d[(d['Location Level'] == 'Country') & (d['tgl'] == akhir)]['Total Cases'].iloc[0])
print(f'Total Cases pada {akhir.date()}: jumlah 34 provinsi = {tp:,} | baris nasional = {tn:,} | selisih = {tn - tp:,} ({(tn - tp) / tn * 100:.2f}%)')

lang = df_tim['Total Deaths'] + df_tim['Total Recovered'] > df_tim['Total Cases']
print()
print('Pelanggaran Total Deaths + Total Recovered > Total Cases per tahun:', lang.groupby(tgl.dt.year).sum().to_dict())
print('Sama persis dengan baris Total Active Cases negatif?', bool((lang == (df_tim['Total Active Cases'] < 0)).all()))
print('Contoh baris pertama dataset:')
print(df_tim.assign(tgl=tgl).sort_values('tgl').head(1)[['Location', 'Date', 'New Cases', 'Total Cases', 'Total Deaths', 'Total Recovered', 'Total Active Cases']].to_string(index=False))

Total Cases pada 2022-09-15: jumlah 34 provinsi = 6,367,805 | baris nasional = 6,402,686 | selisih = 34,881 (0.54%)



Pelanggaran Total Deaths + Total Recovered > Total Cases per tahun: {2020: 371, 2021: 58, 2022: 50}
Sama persis dengan baris Total Active Cases negatif? True
Contoh baris pertama dataset:
   Location     Date  New Cases  Total Cases  Total Deaths  Total Recovered  Total Active Cases
DKI Jakarta 3/1/2020          2           39            20               75                 -56


**Temuan:**
- Tiga aturan yang paling mendasar selalu benar: total kumulatif tidak pernah turun, selisih harinya sama dengan `New Cases`,
  dan kasus aktif = kasus - meninggal - sembuh. Kolom `New Cases` dan `New Deaths` yang dipakai analisis aman.
- Pelanggaran `Total Deaths + Total Recovered > Total Cases` terjadi pada **479 baris, dan barisnya sama persis** dengan
  baris kasus aktif negatif. Tingkat kematian di atas 100% dan kesembuhan di atas 100% adalah bagian dari baris yang sama.
  Jadi ini **satu masalah yang terlihat dari beberapa sisi**, bukan beberapa masalah terpisah. Sebagian besar terjadi pada 2020.
- Jumlah 34 provinsi **tidak sama** dengan baris nasional pada sebagian besar tanggal. Pada hari terakhir selisih totalnya sekitar 0,5%.
  Karena itu baris nasional tidak boleh dicampur dengan baris provinsi.

## 10.3 — Profil kualitas data tim

Semua temuan di atas dirangkum dengan fungsi `catat()` yang sama dengan Bagian 9. Tingkat keparahan ditentukan dari dampaknya
terhadap pertanyaan analisis tim, bukan dari jumlah barisnya.

In [17]:
temuan.clear()   # supaya sel ini aman dijalankan ulang

n_nasional = int(nasional.sum())
n_nol = int((df_tim['New Cases'].eq(0) & provinsi).sum())
n_wilayah = int((df_tim[['Total Rural Villages', 'Total Cities', 'Total Urban Villages']].isna().any(axis=1) & provinsi).sum())
akhir_lokasi = tgl.groupby(df_tim['Location']).max()
n_berakhir_awal = int((akhir_lokasi[akhir_lokasi.index != 'Indonesia'] < tgl[provinsi].max()).sum())
n_langgar = int(lang.sum())

# --- Kelengkapan
catat('Kelengkapan', 'City or Regency', 'Kosong seluruhnya, tidak pernah diisi (struktural)',
      df_tim['City or Regency'].isna().sum(), 'Rendah',
      'Tidak dipakai; buang dari data olahan dan catat sebagai keterbatasan (tidak ada analisis tingkat kota)')
catat('Kelengkapan', 'Special Status', 'Kosong untuk provinsi tanpa status khusus (MAR)',
      df_tim['Special Status'].isna().sum(), 'Rendah', 'Tandai "tidak ada", bukan diisi')
catat('Kelengkapan', 'Growth Factor of New Deaths', 'Kosong bila kemarin 0 dan hari ini lebih dari 0 (dibagi nol) atau hari pertama (MAR)',
      df_tim['Growth Factor of New Deaths'].isna().sum(), 'Rendah', 'Biarkan kosong, nilainya memang tidak terdefinisi; kolom tidak dipakai')
catat('Kelengkapan', 'Growth Factor of New Cases', 'Kosong bila kemarin 0 dan hari ini lebih dari 0 (dibagi nol) atau hari pertama (MAR)',
      df_tim['Growth Factor of New Cases'].isna().sum(), 'Rendah', 'Biarkan kosong, nilainya memang tidak terdefinisi; kolom tidak dipakai')
catat('Kelengkapan', 'Province, Island, Time Zone', 'Kosong pada baris nasional (MAR)',
      df_tim['Island'].isna().sum(), 'Sedang', 'Saring Location Level = Province sebelum mengelompokkan menurut Island')
catat('Kelengkapan', 'Total Rural Villages, Total Cities, Total Urban Villages',
      'Kosong untuk satu provinsi pada tiap kolom (DKI Jakarta, Sulawesi Barat, Aceh)',
      n_wilayah, 'Rendah', 'Tandai tidak berlaku; kolom tidak dipakai')
catat('Kelengkapan', 'New Cases', 'Bernilai 0 pada baris provinsi; nol sejati tidak bisa dibedakan dari hari tanpa pelaporan',
      n_nol, 'Sedang', 'Jangan diubah; total periode tetap benar. Hindari analisis harian pada provinsi kecil, tanyakan ke kompilator')
catat('Kelengkapan', 'Date', 'Data Gorontalo dan Sulawesi Barat berakhir 14 September, provinsi lain 15 September',
      n_berakhir_awal, 'Rendah', 'Catat sebagai keterbatasan; tidak ditambal')
# --- Konsistensi
catat('Konsistensi', 'Location, Location Level', 'Baris nasional (Indonesia) bercampur dengan baris provinsi; menjumlahkan semua baris menghitung ganda',
      n_nasional, 'Tinggi', 'Saring Location Level = Province sebelum agregasi')
# --- Validitas
catat('Validitas', 'Date', 'Tersimpan sebagai teks, bukan tanggal (format bulan/hari/tahun, sudah diverifikasi)',
      len(df_tim), 'Sedang', 'Ubah dengan format %m/%d/%Y lalu periksa rentang bulannya')
catat('Validitas', 'Case Fatality Rate, Case Recovered Rate', 'Tersimpan sebagai teks berformat persen',
      len(df_tim), 'Rendah', 'Ubah ke angka bila dipakai; saat ini tidak dipakai')
catat('Validitas', 'Case Fatality Rate', 'Di atas 100% (bagian dari baris pelanggaran Akurasi di bawah)',
      (cfr > 100).sum(), 'Rendah', 'Jangan dipakai; hitung ulang dari New Deaths dan New Cases')
catat('Validitas', 'Case Recovered Rate', 'Di atas 100% (bagian dari baris pelanggaran Akurasi di bawah)',
      (crr > 100).sum(), 'Rendah', 'Jangan dipakai')
# --- Akurasi
catat('Akurasi', 'Total Deaths, Total Recovered, Total Cases', 'Meninggal + sembuh melebihi total kasus; sama dengan kasus aktif negatif',
      n_langgar, 'Sedang', 'Analisis memakai New Cases dan New Deaths, bukan kolom Total; tandai baris ini dan catat sebagai keterbatasan')
catat('Akurasi', 'New Cases (provinsi vs nasional)', 'Jumlah 34 provinsi tidak sama dengan baris nasional pada tanggal yang sama (baris nasional yang berbeda)',
      tgl_beda, 'Sedang', 'Pakai provinsi saja dan jangan mencampur dengan baris nasional; tanyakan ke kompilator')

profil_tim = pd.DataFrame(temuan).sort_values(['keparahan', 'jumlah'], ascending=[True, False])
print(profil_tim.to_string(index=False))

    dimensi                                                    kolom                                                                                                  masalah  jumlah  persen keparahan                                                                                                       rencana
Kelengkapan                                          City or Regency                                                       Kosong seluruhnya, tidak pernah diisi (struktural)   31822  100.00    Rendah        Tidak dipakai; buang dari data olahan dan catat sebagai keterbatasan (tidak ada analisis tingkat kota)
  Validitas                  Case Fatality Rate, Case Recovered Rate                                                                  Tersimpan sebagai teks berformat persen   31822  100.00    Rendah                                                            Ubah ke angka bila dipakai; saat ini tidak dipakai
Kelengkapan                                           Special Status  

In [18]:
# simpan profil dan lampirannya
profil_tim.to_csv('reports/profil_kualitas_data.csv', index=False, encoding='utf-8')
pola_df.to_csv('reports/pola_kekosongan.csv', index=False, encoding='utf-8')
tabel_aturan.to_csv('reports/aturan_validitas.csv', index=False, encoding='utf-8')
print('Tersimpan di reports/:', sorted(os.listdir('reports')))
print('Profil:', profil_tim.shape[0], 'temuan |', 'Tinggi:', int((profil_tim['keparahan'] == 'Tinggi').sum()),
      '| Sedang:', int((profil_tim['keparahan'] == 'Sedang').sum()), '| Rendah:', int((profil_tim['keparahan'] == 'Rendah').sum()))

Tersimpan di reports/: ['aturan_validitas.csv', 'pola_kekosongan.csv', 'profil_kualitas_data.csv']
Profil: 15 temuan | Tinggi: 1 | Sedang: 5 | Rendah: 9


### Daftar periksa Bagian 10

- [x] Seluruh sembilan pemeriksaan dijalankan pada data tim
- [x] Setiap kolom dengan nilai kosong di atas 5% dinilai MCAR, MAR, atau MNAR, beserta alasannya (`reports/pola_kekosongan.csv`)
- [x] Lebih dari lima aturan validitas ditulis sendiri, dan lebih dari satu aturan hubungan antarkolom diperiksa (`reports/aturan_validitas.csv`)
- [x] Profil kualitas data tersimpan di `reports/profil_kualitas_data.csv`
- [ ] Notebook di-*commit* ke repositori tim dengan pesan yang bermakna